<div style="background:linear-gradient(90deg,#0b3d91,#1f6fb2);color:#fff;padding:22px 26px;border-radius:10px">
<h1 style="margin:0;color:#fff">Python for Data Science — Hands-On Workshop</h1>
<h3 style="margin:6px 0 0;color:#dfe9f7">Session 4 of 4 · End-to-End Data Wrangling</h3>
<p style="margin:10px 0 0">📅 <b>Date:</b> 16/11/2026 &nbsp;|&nbsp; 👤 <b>Instructor:</b> Mr. Mahesh Divakaran &nbsp;|&nbsp; 🏷️ <b>Batch:</b> 30</p>
</div>

### 🎯 What you will learn today

| # | Topic | You will be able to… |
|---|-------|----------------------|
| 1 | Data audit & missing values | find gaps with `isna()`, drop or fill them (mean, median, mode, ffill, group-wise) |
| 2 | Duplicates & data types | `duplicated()`, `drop_duplicates()`, `astype()`, `pd.to_numeric(errors='coerce')` |
| 3 | Text & date cleaning | strip spaces, fix case, extract numbers with regex, parse dates |
| 4 | Grouping | `groupby().agg()` with named results and `groupby().transform()` |
| 5 | Combining tables | `pd.concat()` and `pd.merge()` — inner, left, right, outer |
| 6 | Reshaping | `pivot_table`, `crosstab`, `pivot`, `melt`, `stack`, `unstack` |
| 7 | Capstone pipeline | chain cleaning steps with `.pipe()` and export the result |

### 🧭 How to use this notebook
- Run each cell with **Shift + Enter**, from top to bottom.
- Read the text cell first, then run the code, then read **"What to observe"**.
- Change the numbers and run again — that is the best way to learn!

> ✏️ Theme: the **same 25 patients from Session 3** — but this time the data arrives **messy**, as it often does in real life: extra spaces, mixed case, `"145 mmHg"` text, wrong dates, duplicates and missing values.

## 0. Setup and the Messy Data

First we rebuild Session 3's `visits_df` with the **same seed** (so the numbers match), then we create two "raw" tables from it and **damage them on purpose**:

| Table | One row per | Problems added |
|-------|-------------|----------------|
| `demographics_raw` | patient | spaces & lower-case IDs, mixed sex labels, `"54 yrs"`, `"unknown"`, messy site names, wrong arm spellings, an impossible date, `"kg"` text, duplicates |
| `lab_results_raw` | patient visit | `"WEEK 4"` / `" week 8"` / `"Wk 12"`, `"145 mmHg"`, `"N/A"`, `"high"`, missing HbA1c, duplicate rows, a patient not in demographics |

You don't need to understand every line of this cell — just run it. The cleaning starts in Section 1.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---- 1) Rebuild Session 3's visits_df exactly ----
n_patients, weeks = 25, [0, 4, 8, 12]
patient_ids = [f"P{i:03d}" for i in range(1, n_patients + 1)]
sites = rng.choice(["Kochi", "Pune", "Delhi", "Chennai"], size=n_patients)
ages = rng.integers(28, 76, size=n_patients)
arms = np.where(np.arange(n_patients) % 2 == 0, "Drug A", "Placebo")
start_dates = pd.Timestamp("2026-01-05") + pd.to_timedelta(rng.integers(0, 30, size=n_patients), unit="D")
base_bp = rng.normal(150, 12, size=n_patients)
base_chol = rng.normal(220, 30, size=n_patients)

rows = []
for i in range(n_patients):
    for week in weeks:
        drop = week * 1.2 if arms[i] == "Drug A" else week * 0.2
        rows.append({
            "patient_id": patient_ids[i], "site": sites[i], "visit": f"Week {week}",
            "visit_date": (start_dates[i] + pd.Timedelta(weeks=week)).strftime("%Y-%m-%d"),
            "age": int(ages[i]), "treatment_arm": arms[i],
            "systolic_bp": int(round(base_bp[i] - drop + rng.normal(0, 4))),
            "cholesterol": int(round(base_chol[i] - week * 0.8 + rng.normal(0, 6))),
            "adherence_pct": round(float(rng.uniform(55, 100)), 1),
        })
visits_df = pd.DataFrame(rows)

# ---- 2) demographics_raw: one row per patient ----
week0 = visits_df[visits_df["visit"] == "Week 0"].reset_index(drop=True)
demographics_raw = pd.DataFrame({
    "patient_id": week0["patient_id"],
    "sex": rng.choice(["M", "F"], size=n_patients),
    "age": week0["age"].astype(str),
    "site": week0["site"],
    "treatment_arm": week0["treatment_arm"],
    "enrol_date": pd.to_datetime(week0["visit_date"]).dt.strftime("%d/%m/%Y"),
    "weight": [f"{w:.1f} kg" for w in rng.normal(70, 12, size=n_patients)],
})
d = demographics_raw                                   # short name while we add mess
d.loc[2, "patient_id"] = " p003 "
d.loc[9, "patient_id"] = "p010"
d.loc[[1, 6, 13, 20], "sex"] = ["male", "Female ", " f", np.nan]
d.loc[3, "age"], d.loc[6, "age"], d.loc[11, "age"] = "54 yrs", "unknown", np.nan
d.loc[[1, 5, 8, 14], "site"] = [" chennai", "CHENNAI ", "kochi ", np.nan]
d.loc[[4, 7, 10], "treatment_arm"] = ["drug a", "PLACEBO", "Drug-A"]
d.loc[12, "enrol_date"], d.loc[17, "enrol_date"] = np.nan, "31/02/2026"    # missing + impossible date
d.loc[[2, 15], "weight"] = ["68kg", "N/A"]
d.loc[len(d)] = d.loc[4]                               # exact duplicate of P005
d.loc[len(d)] = ["P010 ", "F", "41", "Kochi", "Placebo", "05/01/2026", "64.0 kg"]   # hidden duplicate
d.loc[len(d)] = ["P026", "M", "39", "Pune", "Drug A", "02/02/2026", "77.3 kg"]     # no lab results

# ---- 3) lab_results_raw: one row per patient visit ----
patient_hba1c = rng.normal(6.4, 0.6, size=n_patients)
lab_results_raw = pd.DataFrame({
    "patient_id": visits_df["patient_id"],
    "site": visits_df["site"],                         # same column name as in demographics
    "visit": visits_df["visit"],
    "sample_date": visits_df["visit_date"],
    "sbp": visits_df["systolic_bp"].astype(str) + " mmHg",
    "cholesterol": visits_df["cholesterol"].astype(str),
    "hba1c": (np.repeat(patient_hba1c, 4) + rng.normal(0, 0.2, size=100)).round(1),
})
lab = lab_results_raw
lab.loc[[5, 22, 47], "visit"] = ["WEEK 4", " week 8", "Wk 12"]
lab.loc[[10, 51], "sbp"] = "N/A"
lab.loc[[3, 30], "sbp"] = lab.loc[[3, 30], "sbp"].str.replace(" ", "")      # "145mmHg"
lab.loc[14, "cholesterol"], lab.loc[70, "cholesterol"] = "", "high"
lab.loc[rng.choice(100, size=8, replace=False), "hba1c"] = np.nan
for r in [7, 33, 60]:
    lab.loc[len(lab)] = lab.loc[r]                     # exact duplicate rows
lab.loc[len(lab)] = ["P027", "Delhi", "Week 0", "2026-01-20", "138 mmHg", "205", 6.1]
lab.loc[len(lab)] = ["P027", "Delhi", "Week 4", "2026-02-17", "131 mmHg", "199", 6.0]

print("demographics_raw:", demographics_raw.shape)
print("lab_results_raw :", lab_results_raw.shape)

demographics_raw: (28, 7)
lab_results_raw : (105, 7)


> 👀 **What to observe:** `demographics_raw` has **28 rows** for 26 real patients (2 duplicates). `lab_results_raw` has **105 rows**: 100 visits + 3 duplicates + 2 rows for P027.

In [2]:
demographics_raw

,patient_id,sex,age,site,treatment_arm,enrol_date,weight
0,P001,M,58,Kochi,Drug A,27/01/2026,62.1 kg
1,P002,male,47,chennai,Placebo,10/01/2026,71.7 kg
2,p003,M,67,Delhi,Drug A,15/01/2026,68kg
3,P004,M,54 yrs,Pune,Placebo,19/01/2026,72.8 kg
4,P005,M,49,Pune,drug a,19/01/2026,76.4 kg
5,P006,M,49,CHENNAI,Placebo,06/01/2026,61.5 kg
6,P007,Female,unknown,Kochi,Drug A,21/01/2026,67.8 kg
7,P008,F,32,Delhi,PLACEBO,09/01/2026,72.4 kg
8,P009,F,54,kochi,Drug A,27/01/2026,79.8 kg
9,p010,F,70,Kochi,Placebo,25/01/2026,65.3 kg


> 👀 **What to observe:** Look for the problems: `' p003 '`, `'p010'`, `'male'`, `' f'`, `'54 yrs'`, `'unknown'`, `' chennai'`, `'drug a'`, `'Drug-A'`, `'31/02/2026'`, `'68kg'`, `'N/A'`, and the duplicate rows at the bottom.

In [3]:
lab_results_raw.head(12)

,patient_id,site,visit,sample_date,sbp,cholesterol,hba1c
0,P001,Kochi,Week 0,2026-01-27,142 mmHg,240,7.1
1,P001,Kochi,Week 4,2026-02-24,135 mmHg,232,6.9
2,P001,Kochi,Week 8,2026-03-24,124 mmHg,223,7.4
3,P001,Kochi,Week 12,2026-04-21,122mmHg,230,7.1
4,P002,Chennai,Week 0,2026-01-10,156 mmHg,249,6.7
5,P002,Chennai,WEEK 4,2026-02-07,160 mmHg,233,NaN
6,P002,Chennai,Week 8,2026-03-07,152 mmHg,233,6.4
7,P002,Chennai,Week 12,2026-04-04,148 mmHg,234,6.5
8,P003,Delhi,Week 0,2026-01-15,157 mmHg,235,6.3
9,P003,Delhi,Week 4,2026-02-12,152 mmHg,242,6.3


> 👀 **What to observe:** `sbp` is text with `' mmHg'`, and some visits are written differently. We will fix all of this step by step.

## 1. Missing Values

### 1.1 Audit First

| Method | Gives |
|--------|-------|
| `df.isna()` | True where a value is missing |
| `df.notna()` | True where a value is present |
| `df.isna().sum()` | missing count per column |
| `df.isna().mean() * 100` | missing % per column |

⚠️ Empty text `""` and words like `"N/A"` or `"unknown"` are **not** counted as missing yet — pandas only sees real `NaN`. We will convert them later.

In [4]:
audit = pd.DataFrame({
    "missing": demographics_raw.isna().sum(),
    "missing_pct": (demographics_raw.isna().mean() * 100).round(1),
})
audit

,missing,missing_pct
patient_id,0,0.0
sex,1,3.6
age,1,3.6
site,1,3.6
treatment_arm,0,0.0
enrol_date,1,3.6
weight,0,0.0


In [5]:
print("Lab missing counts:")
print(lab_results_raw.isna().sum())
print("\nRows with HbA1c present:", lab_results_raw["hba1c"].notna().sum())
print("Rows with any missing value:", lab_results_raw.isna().any(axis=1).sum())

Lab missing counts:
patient_id     0
site           0
visit          0
sample_date    0
sbp            0
cholesterol    0
hba1c          8
dtype: int64

Rows with HbA1c present: 97
Rows with any missing value: 8


> 👀 **What to observe:** Only `hba1c` has real `NaN` in the lab table. `'N/A'` in `sbp` and `''` in `cholesterol` are hidden missing values — they will appear after we convert those columns to numbers.

### 1.2 Dropping Missing Values: `dropna()`

| Option | Meaning |
|--------|---------|
| `dropna()` | drop a row if **any** value is missing |
| `dropna(subset=["col"])` | look only at these columns |
| `dropna(thresh=n)` | keep rows with **at least n** non-missing values |
| `dropna(axis=1)` | drop **columns** instead of rows |

⚠️ Dropping is easy but throws data away. In clinical data, never drop a patient without a reason.

In [6]:
print("Original rows           :", len(demographics_raw))
print("dropna()                :", len(demographics_raw.dropna()))
print("dropna(subset=['site']) :", len(demographics_raw.dropna(subset=["site"])))
print("dropna(thresh=7)        :", len(demographics_raw.dropna(thresh=7)))   # all 7 columns filled
print("dropna(thresh=6)        :", len(demographics_raw.dropna(thresh=6)))   # at most 1 missing

Original rows           : 28
dropna()                : 24
dropna(subset=['site']) : 27
dropna(thresh=7)        : 24
dropna(thresh=6)        : 28


> 👀 **What to observe:** `dropna()` removes every row with any gap. `thresh=6` keeps rows with at most one missing value — a gentler rule.

### 1.3 Filling Missing Values: `fillna()`

| Strategy | Code | Good for |
|----------|------|----------|
| fixed value | `fillna("Unknown")` | text categories |
| mean | `fillna(s.mean())` | numbers without outliers |
| median | `fillna(s.median())` | numbers **with** outliers |
| mode (most common) | `fillna(s.mode()[0])` | categories |
| forward fill | `ffill()` | carry the last value forward in time |
| backward fill | `bfill()` | use the next value |

In [7]:
hba1c = lab_results_raw["hba1c"]

fill_demo = pd.DataFrame({
    "original": hba1c,
    "mean_fill": hba1c.fillna(hba1c.mean()).round(1),
    "median_fill": hba1c.fillna(hba1c.median()),
})
fill_demo[fill_demo["original"].isna()]

,original,mean_fill,median_fill
5,NaN,6.3,6.3
36,NaN,6.3,6.3
37,NaN,6.3,6.3
59,NaN,6.3,6.3
62,NaN,6.3,6.3
71,NaN,6.3,6.3
73,NaN,6.3,6.3
75,NaN,6.3,6.3


In [8]:
site_mode = demographics_raw["site"].mode()[0]
print("Most common site:", site_mode)
print("Missing before:", demographics_raw["site"].isna().sum())
print("Missing after :", demographics_raw["site"].fillna(site_mode).isna().sum())

Most common site: Chennai
Missing before: 1
Missing after : 0


> 👀 **What to observe:** Mean and median fill give the **same number for everyone**, ignoring who the patient is. Mode fill gives the most common category.

### 1.4 Forward Fill and Group-wise Fill

For repeated lab tests, a better guess is the **same patient's** values:
- `groupby("patient_id")["hba1c"].ffill()` → copy the patient's **previous** visit value.
- `groupby("patient_id")["hba1c"].transform("mean")` → the patient's **own average**, then fill with it.

⚠️ Always **sort by patient and date** before `ffill`, and fill **within each patient** — never copy one patient's value to the next.

In [9]:
labs_sorted = lab_results_raw.sort_values(["patient_id", "sample_date"]).copy()

labs_sorted["hba1c_ffill"] = labs_sorted.groupby("patient_id")["hba1c"].ffill()
labs_sorted["hba1c_bfill"] = labs_sorted.groupby("patient_id")["hba1c"].bfill()

patient_mean = labs_sorted.groupby("patient_id")["hba1c"].transform("mean")
labs_sorted["hba1c_group_mean"] = labs_sorted["hba1c"].fillna(patient_mean.round(1))

missing_ids = labs_sorted.loc[labs_sorted["hba1c"].isna(), "patient_id"].unique()
labs_sorted[labs_sorted["patient_id"].isin(missing_ids[:3])][
    ["patient_id", "visit", "hba1c", "hba1c_ffill", "hba1c_bfill", "hba1c_group_mean"]
]

,patient_id,visit,hba1c,hba1c_ffill,hba1c_bfill,hba1c_group_mean
4,P002,Week 0,6.7,6.7,6.7,6.7
5,P002,WEEK 4,NaN,6.7,6.4,6.5
6,P002,Week 8,6.4,6.4,6.4,6.4
7,P002,Week 12,6.5,6.5,6.5,6.5
100,P002,Week 12,6.5,6.5,6.5,6.5
36,P010,Week 0,NaN,NaN,5.6,5.6
37,P010,Week 4,NaN,NaN,5.6,5.6
38,P010,Week 8,5.6,5.6,5.6,5.6
39,P010,Week 12,5.7,5.7,5.7,5.7
56,P015,Week 0,5.6,5.6,5.6,5.6


> 👀 **What to observe:** Each gap is filled from the **same patient's** readings. If the first visit is missing, `ffill` leaves it empty (nothing before it) — `bfill` or the group mean can fill it.

## 2. Duplicates and Data Types

### 2.1 Duplicates

| Method | Does |
|--------|------|
| `df.duplicated()` | True for rows that **repeat** an earlier row |
| `df.duplicated(subset=["id"])` | compare only some columns |
| `df.drop_duplicates()` | remove repeats (keeps the first by default) |
| `keep="last"` / `keep=False` | keep the last / mark **all** copies |

In [10]:
print("Exact duplicate rows in labs        :", lab_results_raw.duplicated().sum())
print("Exact duplicate rows in demographics:", demographics_raw.duplicated().sum())

# Show every copy of the duplicated lab rows
lab_results_raw[lab_results_raw.duplicated(keep=False)].sort_values("patient_id")

Exact duplicate rows in labs        : 3
Exact duplicate rows in demographics: 1


,patient_id,site,visit,sample_date,sbp,cholesterol,hba1c
7,P002,Chennai,Week 12,2026-04-04,148 mmHg,234,6.5
100,P002,Chennai,Week 12,2026-04-04,148 mmHg,234,6.5
33,P009,Kochi,Week 4,2026-02-24,151 mmHg,175,7.2
101,P009,Kochi,Week 4,2026-02-24,151 mmHg,175,7.2
60,P016,Chennai,Week 0,2026-01-14,147 mmHg,241,6.4
102,P016,Chennai,Week 0,2026-01-14,147 mmHg,241,6.4


In [11]:
labs = lab_results_raw.drop_duplicates().reset_index(drop=True)
print("Lab rows after drop_duplicates:", len(labs))

demo = demographics_raw.drop_duplicates().reset_index(drop=True)
print("Demographic rows after drop_duplicates:", len(demo))
print("Duplicate patient_id values:", demo.duplicated(subset=["patient_id"]).sum())

Lab rows after drop_duplicates: 102
Demographic rows after drop_duplicates: 27
Duplicate patient_id values: 0


> 👀 **What to observe:** The 3 exact lab copies and the P005 copy are gone. But `'P010 '` (with a space) is **not** caught yet — it looks different from `'P010'`. We find it after cleaning the text in Section 3.

### 2.2 Converting Types: `astype()` and `pd.to_numeric()`

- `astype(int)` / `astype(float)` → converts, but **crashes** on bad values like `"unknown"`.
- `pd.to_numeric(col, errors="coerce")` → bad values become `NaN` instead of crashing. Then you can **count and review** them.

In [12]:
try:
    demo["age"].astype(float)
except ValueError as error:
    print("astype failed:", error)

age_num = pd.to_numeric(demo["age"], errors="coerce")
print("\nValues that could not be converted:")
print(demo.loc[age_num.isna(), ["patient_id", "age"]])

astype failed: could not convert string to float: '54 yrs'

Values that could not be converted:
   patient_id      age
3        P004   54 yrs
6        P007  unknown
11       P012      NaN


> 👀 **What to observe:** `astype` stops at the first bad value. `to_numeric(errors='coerce')` turns `'54 yrs'`, `'unknown'` and the empty value into `NaN` so we can inspect them. `'54 yrs'` is rescued with regex in Section 3.

In [13]:
labs["cholesterol"] = pd.to_numeric(labs["cholesterol"], errors="coerce")
labs["hba1c"] = labs["hba1c"].astype(float)

print(labs.dtypes)
print("\nCholesterol now missing:", labs["cholesterol"].isna().sum(), "(the '' and 'high' values)")

patient_id         str
site               str
visit              str
sample_date        str
sbp                str
cholesterol    float64
hba1c          float64
dtype: object

Cholesterol now missing: 2 (the '' and 'high' values)


> 👀 **What to observe:** `cholesterol` is now a number column (`float64` — whole-number columns become float when they contain `NaN`).

## 3. Text and Date Cleaning

### 3.1 Cleaning Text with `.str`

| Method | Fixes |
|--------|-------|
| `.str.strip()` | spaces at the start/end |
| `.str.upper()` / `.lower()` / `.title()` | inconsistent case |
| `.str.replace(old, new)` | wrong spellings |
| `.str.extract(r"(\d+)")` | pull numbers out of text (**regex**) |
| `.map(dict)` / `.replace(dict)` | turn many spellings into one standard label |

In [14]:
print("Before:", demo["patient_id"].tolist()[:12])
demo["patient_id"] = demo["patient_id"].str.strip().str.upper()
print("After :", demo["patient_id"].tolist()[:12])

# The hidden duplicate is now visible
print("\nDuplicate IDs after cleaning:", demo.loc[demo.duplicated(subset=["patient_id"], keep=False), "patient_id"].tolist())
demo = demo.drop_duplicates(subset=["patient_id"], keep="first").reset_index(drop=True)
print("Patients now:", len(demo))

Before: ['P001', 'P002', ' p003 ', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'p010', 'P011', 'P012']
After : ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P010', 'P011', 'P012']

Duplicate IDs after cleaning: ['P010', 'P010']
Patients now: 26


> 👀 **What to observe:** After `strip().upper()`, `'P010 '` becomes `'P010'` — a duplicate. **Clean text first, then check duplicates again.** We now have 26 unique patients.

In [15]:
demo["site"] = demo["site"].str.strip().str.title()

sex_map = {"m": "M", "male": "M", "f": "F", "female": "F"}
demo["sex"] = demo["sex"].str.strip().str.lower().map(sex_map)

arm_map = {"drug a": "Drug A", "drug-a": "Drug A", "placebo": "Placebo"}
demo["treatment_arm"] = demo["treatment_arm"].str.strip().str.lower().map(arm_map)

for col in ["site", "sex", "treatment_arm"]:
    print(col, "->", demo[col].value_counts(dropna=False).to_dict())

site -> {'Chennai': 8, 'Kochi': 6, 'Delhi': 6, 'Pune': 5, nan: 1}
sex -> {'F': 13, 'M': 12, nan: 1}
treatment_arm -> {'Drug A': 14, 'Placebo': 12}


> 👀 **What to observe:** Every spelling is now one standard label. `dropna=False` shows the remaining real gaps (`nan`) — they are left missing on purpose rather than guessed.

In [16]:
# Regex: (\d+\.?\d*) = digits, optional dot, optional more digits
demo["age"] = pd.to_numeric(demo["age"].str.extract(r"(\d+)", expand=False), errors="coerce")
demo["weight_kg"] = pd.to_numeric(demo["weight"].str.extract(r"(\d+\.?\d*)", expand=False), errors="coerce")
demo = demo.drop(columns="weight")

labs["sbp"] = pd.to_numeric(labs["sbp"].str.extract(r"(\d+)", expand=False), errors="coerce")

demo[["patient_id", "age", "weight_kg"]].head(5)

,patient_id,age,weight_kg
0,P001,58.0,62.1
1,P002,47.0,71.7
2,P003,67.0,68.0
3,P004,54.0,72.8
4,P005,49.0,76.4


> 👀 **What to observe:** `'54 yrs'` → **54**, `'68kg'` → **68.0**, `'145 mmHg'` → **145**. `'unknown'` and `'N/A'` have no digits, so they become `NaN`.

In [17]:
print("Visit spellings before:", sorted(labs["visit"].unique()))
labs["visit"] = (labs["visit"]
                 .str.strip()
                 .str.title()
                 .str.replace("Wk", "Week", regex=False))
print("Visit spellings after :", sorted(labs["visit"].unique()))
labs["week"] = labs["visit"].str.extract(r"(\d+)", expand=False).astype(int)

Visit spellings before: [' week 8', 'WEEK 4', 'Week 0', 'Week 12', 'Week 4', 'Week 8', 'Wk 12']
Visit spellings after : ['Week 0', 'Week 12', 'Week 4', 'Week 8']


> 👀 **What to observe:** Four spellings collapse into exactly `Week 0 / 4 / 8 / 12`. Several `.str` steps can be chained inside brackets for readability.

### 3.2 Dates with `pd.to_datetime()` and `.dt`

- Give the `format` when you know it: `"%d/%m/%Y"` means day/month/year.
- `errors="coerce"` turns impossible dates (like 31 February) into `NaT` (Not a Time).
- `.dt.year`, `.dt.month`, `.dt.day`, `.dt.day_name()` extract parts.
- Subtracting dates gives a duration; `.dt.days` turns it into a number.

In [18]:
demo["enrol_date"] = pd.to_datetime(demo["enrol_date"], format="%d/%m/%Y", errors="coerce")
labs["sample_date"] = pd.to_datetime(labs["sample_date"], format="%Y-%m-%d")

print("Unparseable or missing enrolment dates:")
print(demo.loc[demo["enrol_date"].isna(), ["patient_id", "enrol_date"]])

# "Int64" (capital I) = whole numbers that can still hold missing values
demo["enrol_year"] = demo["enrol_date"].dt.year.astype("Int64")
demo["enrol_month"] = demo["enrol_date"].dt.month.astype("Int64")
demo["enrol_weekday"] = demo["enrol_date"].dt.day_name()
demo[["patient_id", "enrol_date", "enrol_year", "enrol_month", "enrol_weekday"]].head()

Unparseable or missing enrolment dates:
   patient_id enrol_date
12       P013        NaT
17       P018        NaT


,patient_id,enrol_date,enrol_year,enrol_month,enrol_weekday
0,P001,2026-01-27,2026,1,Tuesday
1,P002,2026-01-10,2026,1,Saturday
2,P003,2026-01-15,2026,1,Thursday
3,P004,2026-01-19,2026,1,Monday
4,P005,2026-01-19,2026,1,Monday


> 👀 **What to observe:** P013 (missing) and P018 (31/02/2026) become `NaT`. Without `errors='coerce'`, the impossible date would crash the whole conversion. `Int64` keeps year/month as whole numbers even with gaps (`<NA>`).

In [19]:
# Days from enrolment to each lab sample (needs both tables -> quick merge preview)
days = labs.merge(demo[["patient_id", "enrol_date"]], on="patient_id", how="left")
days["days_on_study"] = (days["sample_date"] - days["enrol_date"]).dt.days.astype("Int64")
days[["patient_id", "visit", "enrol_date", "sample_date", "days_on_study"]].head(8)

,patient_id,visit,enrol_date,sample_date,days_on_study
0,P001,Week 0,2026-01-27,2026-01-27,0
1,P001,Week 4,2026-01-27,2026-02-24,28
2,P001,Week 8,2026-01-27,2026-03-24,56
3,P001,Week 12,2026-01-27,2026-04-21,84
4,P002,Week 0,2026-01-10,2026-01-10,0
5,P002,Week 4,2026-01-10,2026-02-07,28
6,P002,Week 8,2026-01-10,2026-03-07,56
7,P002,Week 12,2026-01-10,2026-04-04,84


> 👀 **What to observe:** `days_on_study` is 0, 28, 56, 84 — the visit schedule is consistent. (`merge` is explained properly in Section 5.)

In [20]:
print("Clean demographics:")
demo.info()
print("\nClean labs:")
labs.info()

Clean demographics:
<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   patient_id     26 non-null     str           
 1   sex            25 non-null     str           
 2   age            24 non-null     float64       
 3   site           25 non-null     str           
 4   treatment_arm  26 non-null     str           
 5   enrol_date     24 non-null     datetime64[us]
 6   weight_kg      25 non-null     float64       
 7   enrol_year     24 non-null     Int64         
 8   enrol_month    24 non-null     Int64         
 9   enrol_weekday  24 non-null     str           
dtypes: Int64(2), datetime64[us](1), float64(2), str(5)
memory usage: 2.2 KB

Clean labs:
<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----     

> 👀 **What to observe:** All columns now have sensible types: numbers are numbers and dates are dates. The remaining `NaN` are honest gaps.

## 4. Grouping: Split → Apply → Combine

`groupby` **splits** rows into groups, **applies** a calculation to each group, and **combines** the results.

| Pattern | Result |
|---------|--------|
| `df.groupby("visit")["sbp"].mean()` | one number per group |
| `.agg(new_name=("column", "function"))` | several **named** results at once |
| `.transform("mean")` | the group result **copied back to every row** (same length as `df`) |

In [21]:
print(labs.groupby("visit")["sbp"].mean().round(1))

visit
Week 0     149.8
Week 12    142.0
Week 4     146.6
Week 8     144.8
Name: sbp, dtype: float64


> 👀 **What to observe:** Groups are sorted alphabetically as text (`Week 0, Week 12, Week 4, Week 8`). Grouping by the numeric `week` column gives the natural order.

In [22]:
visit_summary = (labs.groupby("week")
                 .agg(n_samples=("patient_id", "count"),
                      n_patients=("patient_id", "nunique"),
                      mean_sbp=("sbp", "mean"),
                      median_chol=("cholesterol", "median"),
                      max_hba1c=("hba1c", "max"),
                      missing_hba1c=("hba1c", lambda s: s.isna().sum()))
                 .round(1))
visit_summary

,n_samples,n_patients,mean_sbp,median_chol,max_hba1c,missing_hba1c
week,,,,,,
0,26,26,149.8,212.5,7.6,1
4,26,26,146.6,213.0,7.2,3
8,25,25,144.8,210.0,7.4,1
12,25,25,142.0,207.0,8.0,3


> 👀 **What to observe:** Named aggregation gives **clear column names** in one step. A `lambda` can be used for a custom calculation like counting missing values.

In [23]:
# transform: compare each reading with the patient's own baseline (Week 0)
labs = labs.sort_values(["patient_id", "week"]).reset_index(drop=True)
labs["sbp_baseline"] = labs.groupby("patient_id")["sbp"].transform("first")
labs["sbp_change"] = labs["sbp"] - labs["sbp_baseline"]

# transform: z-score within each visit
labs["chol_z"] = ((labs["cholesterol"] - labs.groupby("week")["cholesterol"].transform("mean"))
                  / labs.groupby("week")["cholesterol"].transform("std")).round(2)

labs[["patient_id", "week", "sbp", "sbp_baseline", "sbp_change", "chol_z"]].head(8)

,patient_id,week,sbp,sbp_baseline,sbp_change,chol_z
0,P001,0,142.0,142.0,0.0,1.11
1,P001,4,135.0,142.0,-7.0,0.99
2,P001,8,124.0,142.0,-18.0,0.56
3,P001,12,122.0,142.0,-20.0,1.16
4,P002,0,156.0,156.0,0.0,1.54
5,P002,4,160.0,156.0,4.0,1.04
6,P002,8,152.0,156.0,-4.0,1.03
7,P002,12,148.0,156.0,-8.0,1.35


> 👀 **What to observe:** `transform` keeps **one value per row**, so it can be stored as a new column. `agg` shrinks to **one row per group**. (`'first'` skips missing values, so it uses the first recorded SBP.)

## 5. Combining Tables

### 5.1 `pd.concat()` — Stack Tables

Use `concat` when tables have the **same columns** (for example, a new batch of patients). `ignore_index=True` renumbers the rows.

In [24]:
late_enrolments = pd.DataFrame({
    "patient_id": ["P028", "P029"],
    "sex": ["F", "M"],
    "age": [52, 61],
    "site": ["Kochi", "Delhi"],
    "treatment_arm": ["Placebo", "Drug A"],
    "enrol_date": pd.to_datetime(["2026-03-02", "2026-03-09"]),
})
combined = pd.concat([demo, late_enrolments], ignore_index=True)
print("demo:", len(demo), "+ late:", len(late_enrolments), "=", len(combined))
combined.tail(4)

demo: 26 + late: 2 = 28


,patient_id,sex,age,site,treatment_arm,enrol_date,weight_kg,enrol_year,enrol_month,enrol_weekday
24,P025,F,65.0,Chennai,Drug A,2026-01-25,69.2,2026,1,Sunday
25,P026,M,39.0,Pune,Drug A,2026-02-02,77.3,2026,2,Monday
26,P028,F,52.0,Kochi,Placebo,2026-03-02,NaN,<NA>,<NA>,NaN
27,P029,M,61.0,Delhi,Drug A,2026-03-09,NaN,<NA>,<NA>,NaN


> 👀 **What to observe:** Columns that exist only in `demo` (like `weight_kg`) are filled with `NaN` for the new rows. We keep using `demo` (without these two patients) below.

### 5.2 `pd.merge()` — Join Tables on a Key

```
demo (1 row per patient)  +  labs (4 rows per patient)  →  joined on patient_id
```

| `how=` | Keeps | Picture |
|--------|-------|---------|
| `"inner"` | only IDs found in **both** | overlap only |
| `"left"` | all rows of the **left** table | left + matches |
| `"right"` | all rows of the **right** table | right + matches |
| `"outer"` | **everything** from both | union |

Helpful options:
- `indicator=True` adds a `_merge` column: `both`, `left_only`, `right_only`.
- `validate="one_to_many"` **checks** the key is unique on the left (catches duplicate IDs).
- `suffixes=("_demo", "_lab")` renames columns that exist in both tables.

In [25]:
for how in ["inner", "left", "right", "outer"]:
    joined = pd.merge(demo, labs, on="patient_id", how=how)
    print(f"{how:<6}: {len(joined):>3} rows | patients: {joined['patient_id'].nunique()}")

inner : 100 rows | patients: 25
left  : 101 rows | patients: 26
right : 102 rows | patients: 26
outer : 103 rows | patients: 27


> 👀 **What to observe:** **P026** has demographics but no labs; **P027** has labs but no demographics. So `left` keeps P026, `right` keeps P027, `outer` keeps both, `inner` keeps neither.

In [26]:
check = pd.merge(demo[["patient_id", "site"]], labs[["patient_id", "site", "visit"]],
                 on="patient_id", how="outer", indicator=True,
                 suffixes=("_demo", "_lab"))

print(check["_merge"].value_counts())
check[check["_merge"] != "both"]

_merge
both          100
right_only      2
left_only       1
Name: count, dtype: int64


,patient_id,site_demo,site_lab,visit,_merge
100,P026,Pune,NaN,NaN,left_only
101,P027,NaN,Delhi,Week 0,right_only
102,P027,NaN,Delhi,Week 4,right_only


> 👀 **What to observe:** `indicator=True` shows exactly which records did not match — useful for data queries back to sites. `site` appears in both tables, so pandas adds the `_demo` / `_lab` suffixes.

In [27]:
# validate catches duplicate keys before they multiply your rows
dup_demo = pd.concat([demo, demo.iloc[[0]]])             # P001 twice on purpose
try:
    pd.merge(dup_demo, labs, on="patient_id", how="left", validate="one_to_many")
except pd.errors.MergeError as error:
    print("MergeError:", error)

analysis_df = pd.merge(demo, labs.drop(columns="site"), on="patient_id",
                       how="inner", validate="one_to_many")
print("\nanalysis_df:", analysis_df.shape)

MergeError: Merge keys are not unique in left dataset; not a one-to-many merge
Duplicates in left:
 patient_id
      P001 ...

analysis_df: (100, 19)


> 👀 **What to observe:** `validate` raised an error because P001 appeared twice on the left. Our real `analysis_df` (inner join, lab `site` dropped) has **100 rows** — 25 patients × 4 visits.

## 6. Reshaping: Long ↔ Wide

| Shape | Looks like | Good for |
|-------|-----------|----------|
| **Long** | one row per patient **per visit** | groupby, plotting, statistics |
| **Wide** | one row per patient, one **column per visit** | reports, Excel, change calculations |

| Tool | Does |
|------|------|
| `pivot_table(index, columns, values, aggfunc)` | long → wide **with** aggregation |
| `pd.crosstab(a, b)` | count table of two categories |
| `pivot(index, columns, values)` | long → wide, **no** aggregation (each cell must be unique) |
| `melt(id_vars, value_vars)` | wide → long |
| `stack()` / `unstack()` | move column labels to the index / back |

In [28]:
pd.pivot_table(analysis_df, index="treatment_arm", columns="week",
               values="sbp", aggfunc="mean").round(1)

week,0,4,8,12
treatment_arm,,,,
Drug A,147.0,142.9,136.9,132.8
Placebo,153.9,151.9,152.6,151.2


> 👀 **What to observe:** A classic results table: Drug A falls from about 147 to about 133 mmHg; Placebo stays near its starting level.

In [29]:
pd.crosstab(demo["sex"], demo["treatment_arm"], margins=True, dropna=False)

treatment_arm,Drug A,Placebo,All
sex,,,
F,6,7,13
M,7,5,12
NaN,1,0,1
All,14,12,26


> 👀 **What to observe:** `crosstab` counts patients for each sex × arm combination. `margins=True` adds the **All** totals.

In [30]:
sbp_wide = analysis_df.pivot(index="patient_id", columns="week", values="sbp")
sbp_wide.columns = [f"sbp_wk{c}" for c in sbp_wide.columns]      # nicer names
sbp_wide = sbp_wide.reset_index()
sbp_wide["change_wk12"] = sbp_wide["sbp_wk12"] - sbp_wide["sbp_wk0"]
sbp_wide.head()

,patient_id,sbp_wk0,sbp_wk4,sbp_wk8,sbp_wk12,change_wk12
0,P001,142.0,135.0,124.0,122.0,-20.0
1,P002,156.0,160.0,152.0,148.0,-8.0
2,P003,157.0,152.0,NaN,145.0,-12.0
3,P004,160.0,151.0,153.0,153.0,-7.0
4,P005,143.0,149.0,124.0,126.0,-17.0


In [31]:
sbp_long = sbp_wide.melt(id_vars="patient_id",
                         value_vars=["sbp_wk0", "sbp_wk4", "sbp_wk8", "sbp_wk12"],
                         var_name="visit", value_name="sbp")
print("Wide:", sbp_wide.shape, "-> Long:", sbp_long.shape)
sbp_long.sort_values(["patient_id", "visit"]).head(6)

Wide: (25, 6) -> Long: (100, 3)


,patient_id,visit,sbp
0,P001,sbp_wk0,142.0
75,P001,sbp_wk12,122.0
25,P001,sbp_wk4,135.0
50,P001,sbp_wk8,124.0
1,P002,sbp_wk0,156.0
76,P002,sbp_wk12,148.0


> 👀 **What to observe:** `pivot` gave **one row per patient** (easy change calculation). `melt` turned it back into **4 rows per patient**.

In [32]:
arm_week = analysis_df.groupby(["treatment_arm", "week"])["sbp"].mean().round(1)
print("Long Series with a 2-level index:")
print(arm_week.head(4))

print("\nunstack() -> weeks become columns:")
wide = arm_week.unstack()
print(wide)

print("\nstack() -> back to long:")
print(wide.stack().head(4))

Long Series with a 2-level index:
treatment_arm  week
Drug A         0       147.0
               4       142.9
               8       136.9
               12      132.8
Name: sbp, dtype: float64

unstack() -> weeks become columns:
week              0      4      8      12
treatment_arm                            
Drug A         147.0  142.9  136.9  132.8
Placebo        153.9  151.9  152.6  151.2

stack() -> back to long:
treatment_arm  week
Drug A         0       147.0
               4       142.9
               8       136.9
               12      132.8
dtype: float64


> 👀 **What to observe:** `unstack()` moves the inner index level (`week`) into columns; `stack()` moves it back. This is the same result as `pivot_table` above.

## 7. Capstone: A Clean, Repeatable Pipeline with `.pipe()`

So far we cleaned step by step. In real projects, put each step in a **small function** and chain them:

```python
result = raw_df.pipe(step_1).pipe(step_2).pipe(step_3)
```

**Why:** reads top-to-bottom like a recipe, each step can be tested alone, and the whole pipeline re-runs on next month's data in one line.

In [33]:
def standardise_text(df, cols):
    df = df.copy()
    for col in cols:
        df[col] = df[col].str.strip()
    return df


def clean_demographics(df):
    df = df.copy()
    df["patient_id"] = df["patient_id"].str.upper()
    df["sex"] = df["sex"].str.lower().map({"m": "M", "male": "M", "f": "F", "female": "F"})
    df["site"] = df["site"].str.title()
    df["treatment_arm"] = df["treatment_arm"].str.lower().map(
        {"drug a": "Drug A", "drug-a": "Drug A", "placebo": "Placebo"})
    df["age"] = pd.to_numeric(df["age"].str.extract(r"(\d+)", expand=False), errors="coerce")
    df["weight_kg"] = pd.to_numeric(df["weight"].str.extract(r"(\d+\.?\d*)", expand=False), errors="coerce")
    df["enrol_date"] = pd.to_datetime(df["enrol_date"], format="%d/%m/%Y", errors="coerce")
    return df.drop(columns="weight").drop_duplicates(subset="patient_id").reset_index(drop=True)


def clean_labs(df):
    df = df.drop_duplicates().copy()
    df["visit"] = df["visit"].str.title().str.replace("Wk", "Week", regex=False)
    df["week"] = df["visit"].str.extract(r"(\d+)", expand=False).astype(int)
    df["sbp"] = pd.to_numeric(df["sbp"].str.extract(r"(\d+)", expand=False), errors="coerce")
    df["cholesterol"] = pd.to_numeric(df["cholesterol"], errors="coerce")
    df["sample_date"] = pd.to_datetime(df["sample_date"])
    df = df.sort_values(["patient_id", "week"])
    df["hba1c"] = df["hba1c"].fillna(df.groupby("patient_id")["hba1c"].transform("mean").round(1))
    return df.drop(columns="site").reset_index(drop=True)


def add_derived(df):
    df = df.copy()
    df["sbp_change"] = df["sbp"] - df.groupby("patient_id")["sbp"].transform("first")
    df["bp_controlled"] = df["sbp"] < 140
    return df


demo_clean = demographics_raw.pipe(standardise_text, cols=["patient_id", "sex", "site", "treatment_arm"]) \
                             .pipe(clean_demographics)
labs_clean = lab_results_raw.pipe(standardise_text, cols=["patient_id", "visit"]).pipe(clean_labs)

final_df = (demo_clean
            .merge(labs_clean, on="patient_id", how="inner", validate="one_to_many")
            .pipe(add_derived))

print("demo_clean:", demo_clean.shape, "| labs_clean:", labs_clean.shape, "| final_df:", final_df.shape)

demo_clean: (26, 7) | labs_clean: (102, 7) | final_df: (100, 15)


> 👀 **What to observe:** The same cleaning as Sections 1–5, now in **four reusable functions**. `final_df` has 100 rows — one per patient visit.

In [34]:
summary_table = (final_df[final_df["week"] == 12]
                 .groupby("treatment_arm")
                 .agg(patients=("patient_id", "nunique"),
                      mean_age=("age", "mean"),
                      mean_sbp_wk12=("sbp", "mean"),
                      mean_sbp_change=("sbp_change", "mean"),
                      pct_controlled=("bp_controlled", "mean"),
                      mean_hba1c=("hba1c", "mean"))
                 .round(2))
summary_table["pct_controlled"] = (summary_table["pct_controlled"] * 100).round(1)
summary_table

,patients,mean_age,mean_sbp_wk12,mean_sbp_change,pct_controlled,mean_hba1c
treatment_arm,,,,,,
Drug A,13,58.50,132.75,-14.17,62.0,6.44
Placebo,12,48.09,151.25,-2.67,8.0,6.23


> 👀 **What to observe:** An **analysis-ready summary**: Drug A shows a larger SBP fall and a higher % of patients below 140 mmHg at Week 12.

### 7.1 Exporting the Results

- `to_csv()` — plain text, opens anywhere.
- `to_excel()` — needs the `openpyxl` package (pre-installed in Colab and Anaconda). Several sheets can go into one file with `pd.ExcelWriter`.

In [35]:
from pathlib import Path

out_dir = Path("session4_outputs")
out_dir.mkdir(exist_ok=True)

final_df.to_csv(out_dir / "analysis_ready_visits.csv", index=False)
summary_table.to_csv(out_dir / "week12_summary.csv")
print("CSV files saved:", sorted(p.name for p in out_dir.glob("*.csv")))

try:
    with pd.ExcelWriter(out_dir / "session4_results.xlsx") as writer:
        final_df.to_excel(writer, sheet_name="visits", index=False)
        summary_table.to_excel(writer, sheet_name="week12_summary")
        sbp_wide.to_excel(writer, sheet_name="sbp_wide", index=False)
    print("Excel file saved: session4_results.xlsx (3 sheets)")
except ImportError:
    print("Install openpyxl to save Excel files:  pip install openpyxl")

CSV files saved: ['analysis_ready_visits.csv', 'week12_summary.csv']
Excel file saved: session4_results.xlsx (3 sheets)


> 👀 **What to observe:** A folder `session4_outputs` appears next to this notebook with two CSV files and one Excel workbook with three sheets.

---
## 🧪 Practice Exercises

### Exercise 1 — Beginner: Medication table audit
```python
meds_raw = pd.DataFrame({
    "patient_id": ["P001", "P002", "P002", "P003", "P004", "P005"],
    "drug": ["Metformin", "atorvastatin ", "atorvastatin ", " METFORMIN", "Aspirin", np.nan],
    "dose": ["500 mg", "20mg", "20mg", "1000 mg", "75 mg", "10 mg"],
})
```
1. Count missing values per column and the number of exact duplicate rows.
2. Drop the duplicates.
3. Clean `drug` (strip spaces, title case) and fill the missing drug with `"Unknown"`.
4. Create `dose_mg` as a number using `.str.extract` and `pd.to_numeric`.

In [36]:
# Exercise 1 - write your code here

### Exercise 2 — Intermediate: Group summary and transform
Using `final_df`:
1. Build a table by `site` with named aggregations: number of patients, mean SBP, and mean cholesterol (round to 1).
2. Add a column `sbp_vs_site_mean` = each visit's SBP minus its **site's** mean SBP (use `transform`).
3. Show the 5 visits with the largest positive `sbp_vs_site_mean`.

In [37]:
# Exercise 2 - write your code here

### Exercise 3 — Challenge: Adverse events pipeline
```python
ae_raw = pd.DataFrame({
    "patient_id": ["p001", "P002 ", "P004", "P004", "P007", "P027", "P010", "P012"],
    "event": ["Headache", "nausea", "Dizziness", "Dizziness", "HEADACHE", "Rash", "Nausea ", "headache"],
    "severity": ["Mild", "moderate", "MILD", "MILD", "Severe", "Mild", "mild", "Moderate"],
    "onset": ["12/02/2026", "20/02/2026", "03/03/2026", "03/03/2026", "15/02/2026", "01/03/2026", "31/02/2026", "28/02/2026"],
})
```
1. Write a function `clean_ae(df)` that strips/standardises `patient_id`, `event` (title case) and `severity` (title case), parses `onset` with `errors="coerce"`, and drops exact duplicates.
2. Run it with `.pipe()`.
3. Merge with `demo_clean` using a **left** join from the AE table and `indicator=True`. Which AE patient has no demographics?
4. Make a `pd.crosstab` of `treatment_arm` × `severity`, and a `pivot_table` counting events per `event` × `treatment_arm`.

In [38]:
# Exercise 3 - write your code here

---
## ✅ Solutions
The solution cells are **collapsed** — open them after you try.

In [39]:
#@title Solution 1 — Medication table audit
meds_raw = pd.DataFrame({
    "patient_id": ["P001", "P002", "P002", "P003", "P004", "P005"],
    "drug": ["Metformin", "atorvastatin ", "atorvastatin ", " METFORMIN", "Aspirin", np.nan],
    "dose": ["500 mg", "20mg", "20mg", "1000 mg", "75 mg", "10 mg"],
})

print("Missing per column:")
print(meds_raw.isna().sum())
print("Duplicate rows:", meds_raw.duplicated().sum())

meds = meds_raw.drop_duplicates().copy()
meds["drug"] = meds["drug"].str.strip().str.title().fillna("Unknown")
meds["dose_mg"] = pd.to_numeric(meds["dose"].str.extract(r"(\d+)", expand=False), errors="coerce")
meds

Missing per column:
patient_id    0
drug          1
dose          0
dtype: int64
Duplicate rows: 1


,patient_id,drug,dose,dose_mg
0,P001,Metformin,500 mg,500
1,P002,Atorvastatin,20mg,20
3,P003,Metformin,1000 mg,1000
4,P004,Aspirin,75 mg,75
5,P005,Unknown,10 mg,10


In [40]:
#@title Solution 2 — Group summary and transform
site_table = (final_df.groupby("site")
              .agg(patients=("patient_id", "nunique"),
                   mean_sbp=("sbp", "mean"),
                   mean_chol=("cholesterol", "mean"))
              .round(1))
print(site_table)

ex2 = final_df.copy()
ex2["sbp_vs_site_mean"] = (ex2["sbp"] - ex2.groupby("site")["sbp"].transform("mean")).round(1)
ex2.nlargest(5, "sbp_vs_site_mean")[["patient_id", "site", "week", "sbp", "sbp_vs_site_mean"]]

         patients  mean_sbp  mean_chol
site                                  
Chennai         8     149.7      217.6
Delhi           6     145.0      214.5
Kochi           6     140.8      210.3
Pune            4     154.2      196.0


,patient_id,site,week,sbp,sbp_vs_site_mean
32,P009,Kochi,0,162.0,21.2
76,P020,Pune,0,175.0,20.8
54,P014,Chennai,8,168.0,18.3
34,P009,Kochi,8,158.0,17.2
78,P020,Pune,8,170.0,15.8


In [41]:
#@title Solution 3 — Adverse events pipeline
ae_raw = pd.DataFrame({
    "patient_id": ["p001", "P002 ", "P004", "P004", "P007", "P027", "P010", "P012"],
    "event": ["Headache", "nausea", "Dizziness", "Dizziness", "HEADACHE", "Rash", "Nausea ", "headache"],
    "severity": ["Mild", "moderate", "MILD", "MILD", "Severe", "Mild", "mild", "Moderate"],
    "onset": ["12/02/2026", "20/02/2026", "03/03/2026", "03/03/2026", "15/02/2026", "01/03/2026", "31/02/2026", "28/02/2026"],
})

def clean_ae(df):
    df = df.copy()
    df["patient_id"] = df["patient_id"].str.strip().str.upper()
    df["event"] = df["event"].str.strip().str.title()
    df["severity"] = df["severity"].str.strip().str.title()
    df["onset"] = pd.to_datetime(df["onset"], format="%d/%m/%Y", errors="coerce")
    return df.drop_duplicates().reset_index(drop=True)

ae = ae_raw.pipe(clean_ae)
print("Clean AE rows:", len(ae), "| bad onset dates:", ae["onset"].isna().sum())

ae_demo = ae.merge(demo_clean[["patient_id", "treatment_arm"]], on="patient_id",
                   how="left", indicator=True)
print("No demographics for:", ae_demo.loc[ae_demo["_merge"] == "left_only", "patient_id"].tolist())

print("\nArm x severity:")
print(pd.crosstab(ae_demo["treatment_arm"], ae_demo["severity"]))

print("\nEvents per arm:")
print(pd.pivot_table(ae_demo, index="event", columns="treatment_arm",
                     values="patient_id", aggfunc="count", fill_value=0))

Clean AE rows: 7 | bad onset dates: 1
No demographics for: ['P027']

Arm x severity:
severity       Mild  Moderate  Severe
treatment_arm                        
Drug A            1         0       1
Placebo           2         2       0

Events per arm:
treatment_arm  Drug A  Placebo
event                         
Dizziness           0        1
Headache            2        1
Nausea              0        2


---
### 📌 Session 4 Summary

| Step | Tools |
|------|-------|
| Audit | `isna().sum()`, `duplicated()`, `info()`, `value_counts(dropna=False)` |
| Missing | `dropna(subset, thresh)`, `fillna(mean/median/mode)`, `groupby().ffill()`, group-wise `transform` |
| Types | `pd.to_numeric(errors="coerce")`, `astype()`, `pd.to_datetime(format, errors="coerce")` |
| Text | `.str.strip/upper/title/replace/extract`, `.map(dict)` |
| Group | `groupby().agg(name=("col", "func"))`, `groupby().transform()` |
| Combine | `pd.concat`, `pd.merge(how, indicator, validate, suffixes)` |
| Reshape | `pivot_table`, `crosstab`, `pivot`, `melt`, `stack`, `unstack` |
| Pipeline | small functions + `.pipe()` → `to_csv` / `to_excel` |

### 🎓 Workshop Complete!
Session 1 Python basics → Session 2 NumPy arrays → Session 3 Pandas tables → Session 4 clean, merged, analysis-ready data.
**Golden rule of data wrangling:** *audit → clean text → fix types → check duplicates again → merge with validation → summarise.*